# Customer Segmentation

## Trial 1

In [18]:
import os

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max as spark_max, sum as spark_sum, count, datediff, to_date, lit
import warnings
warnings.filterwarnings('ignore')

In [20]:
spark = SparkSession.builder \
    .appName("CIMBCustomerSegmentation") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Session Created.")

Spark Session Created.


In [21]:
df_spark = spark.read.csv("ecommerce_dataset.csv", header=True, inferSchema=True)

COL_CUSTOMER = "customer_id"  
COL_DATE = "order_date"       
COL_MONETARY = "total_price_usd"

In [22]:
df_spark = df_spark.withColumn("Order_Date_Formatted", to_date(col(COL_DATE)))

# Dapatkan tanggal transaksi terakhir di seluruh dataset sebagai titik acuan 
max_date = df_spark.agg(spark_max("Order_Date_Formatted")).collect()[0][0]

# Agregasi data level transaksi menjadi level nasabah/pelanggan
rfm_spark = df_spark.groupBy(COL_CUSTOMER).agg(
    spark_max("Order_Date_Formatted").alias("Last_Transaction_Date"),
    count(COL_CUSTOMER).alias("Frequency"),
    spark_sum(COL_MONETARY).alias("Monetary")
)

# Hitung Recency (Selisih hari dari transaksi terakhir pelanggan dengan transaksi terakhir di dataset)
rfm_spark = rfm_spark.withColumn(
    "Recency", 
    datediff(lit(max_date), col("Last_Transaction_Date"))
)

In [23]:
rfm_spark = rfm_spark.drop("Last_Transaction_Date")

rfm_spark = rfm_spark.withColumnRenamed(COL_CUSTOMER, "Customer_ID")

print("=== HASIL AGREGASI RFM ===")
rfm_spark.show(5)

print("=== HASIL AGREGASI RFM ===")
rfm_spark.show(5)

=== HASIL AGREGASI RFM ===
+-----------+---------+--------+-------+
|Customer_ID|Frequency|Monetary|Recency|
+-----------+---------+--------+-------+
|  CUS-9F6Y5|        1|  503.26|    378|
|  CUS-8FMW4|        1|  206.32|    202|
|  CUS-V17TZ|        1|  244.54|    125|
|  CUS-NNQPN|        1|  177.54|    392|
|  CUS-2ASFL|        1|  328.55|     80|
+-----------+---------+--------+-------+
only showing top 5 rows
=== HASIL AGREGASI RFM ===
+-----------+---------+--------+-------+
|Customer_ID|Frequency|Monetary|Recency|
+-----------+---------+--------+-------+
|  CUS-9F6Y5|        1|  503.26|    378|
|  CUS-8FMW4|        1|  206.32|    202|
|  CUS-V17TZ|        1|  244.54|    125|
|  CUS-NNQPN|        1|  177.54|    392|
|  CUS-2ASFL|        1|  328.55|     80|
+-----------+---------+--------+-------+
only showing top 5 rows


In [24]:
rfm_pd = rfm_spark.toPandas()

os.makedirs("data/processed", exist_ok=True)

# Simpan file CSV 
rfm_pd.to_csv("data/processed/rfm_features.csv", index=False)

print("Data processing selesai.")
spark.stop()

Data processing selesai.


## Trial 2

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max as spark_max, sum as spark_sum, count, datediff, to_date, lit, avg, when, expr
import os
import warnings

warnings.filterwarnings('ignore')

In [26]:
spark = SparkSession.builder \
    .appName("CIMB_Behavioral_Segmentation") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Session Created.")

df_spark = spark.read.csv("ecommerce_dataset.csv", header=True, inferSchema=True)
df_spark = df_spark.withColumn("Order_Date_Formatted", to_date(col("order_date")))

Spark Session Created.


In [27]:
# Standarisasi kupon
df_spark = df_spark.withColumn(
    "coupon_used_int", 
    when(col("coupon_used") == "Yes", 1)
    .when(col("coupon_used") == "True", 1)
    .when(col("coupon_used") == "1", 1) 
    .otherwise(0)
)

In [28]:
# bersihkan kolom angka menggunakan fungsi try_cast bawaan SQL Spark
kolom_angka = ["discount_percent", "session_duration_minutes", "total_price_usd"]

for c in kolom_angka:
    df_spark = df_spark.withColumn(c, expr(f"try_cast({c} as double)"))
    # Isi nilai NULL dengan angka 0
    df_spark = df_spark.fillna(0, subset=[c])

# tanggal transaksi terakhir
max_date = df_spark.agg(spark_max("Order_Date_Formatted")).collect()[0][0]

In [29]:
# Agregasi data level transaksi menjadi level nasabah/pelanggan
rfm_spark = df_spark.groupBy("customer_id").agg(
    spark_max("Order_Date_Formatted").alias("Last_Transaction_Date"),
    count("customer_id").alias("Frequency"),
    spark_sum("total_price_usd").alias("Monetary"),
    
    # Fitur Perilaku Tambahan
    avg("discount_percent").alias("Avg_Discount_Percent"),
    avg("session_duration_minutes").alias("Avg_Session_Duration"),
    avg("coupon_used_int").alias("Coupon_Usage_Rate")
)

# Hitung Recency
rfm_spark = rfm_spark.withColumn(
    "Recency", 
    datediff(lit(max_date), col("Last_Transaction_Date"))
)

In [30]:
rfm_spark = rfm_spark.drop("Last_Transaction_Date")
rfm_spark = rfm_spark.withColumnRenamed("customer_id", "Customer_ID")

print("=== HASIL AGREGASI RFM + BEHAVIORAL ===")
rfm_spark.show(5)

=== HASIL AGREGASI RFM + BEHAVIORAL ===
+-----------+---------+--------+--------------------+--------------------+-----------------+-------+
|Customer_ID|Frequency|Monetary|Avg_Discount_Percent|Avg_Session_Duration|Coupon_Usage_Rate|Recency|
+-----------+---------+--------+--------------------+--------------------+-----------------+-------+
|  CUS-9F6Y5|        1|  503.26|                 5.0|                24.1|              0.0|    378|
|  CUS-8FMW4|        1|  206.32|                 0.0|                 6.3|              1.0|    202|
|  CUS-V17TZ|        1|  244.54|                 0.0|                 7.7|              0.0|    125|
|  CUS-NNQPN|        1|  177.54|                 5.0|                10.6|              0.0|    392|
|  CUS-2ASFL|        1|  328.55|                10.0|                16.7|              0.0|     80|
+-----------+---------+--------+--------------------+--------------------+-----------------+-------+
only showing top 5 rows


In [31]:
rfm_pd = rfm_spark.toPandas()

# Simpan ke CSV
os.makedirs("data/processed", exist_ok=True)
rfm_pd.to_csv("data/processed/rfm_behavioral_features.csv", index=False)

print("Data processing selesai.")
spark.stop()

Data processing selesai.
